# 10 — Rank-Based Snapshot Verification Backtest

## Purpose

This notebook independently verifies the exact classification rules implemented
in `09_rank_based_prediction_snapshot.ipynb`.

Notebook 08 established that cross-sectional downside-risk ranks produced more
stable historical risk separation than fixed downside-probability bands.
Notebook 09 implemented that finding in the version 2 latest prediction
snapshot.

This notebook applies the exact version 2 rules to historical monthly
cross-sections and checks whether the implementation reproduces the historical
results.

## Version Lineage

| Stage | Notebook | Role |
|---|---|---|
| Original snapshot | `07_latest_prediction_snapshot.ipynb` | Fixed-band latest prediction snapshot |
| Initial validation | `08_historical_signal_backtest.ipynb` | Historical fixed-versus-rank research |
| Version 2 snapshot | `09_rank_based_prediction_snapshot.ipynb` | Rank-based latest prediction snapshot |
| Implementation verification | `10_rank_based_snapshot_verification_backtest.ipynb` | Exact historical verification of v2 rules |

## Rules Being Verified

### Opportunity Tiers

| Outperform rank | Opportunity tier |
|---:|---|
| 1–18 | High Opportunity |
| 19–45 | Moderate Opportunity |
| 46–90 | Low Opportunity |

### Detailed Downside-Risk Quintiles

| Downside-risk rank | Relative risk quintile |
|---:|---|
| 1–18 | Q1 — Highest Risk |
| 19–36 | Q2 — High Risk |
| 37–54 | Q3 — Medium Risk |
| 55–72 | Q4 — Lower Risk |
| 73–90 | Q5 — Lowest Risk |

### Broad Relative-Risk Groups

| Downside-risk rank | Broad risk group |
|---:|---|
| 1–45 | Higher Relative Risk |
| 46–90 | Lower Relative Risk |

The broad risk group is combined with the opportunity tier to create the six
version 2 opportunity-risk classes.

## Main Verification Questions

1. Can we reproduce the same 30 monthly rebalance dates and 2,700 observations
   used in Notebook 08?
2. Does every rebalance date contain 90 eligible stocks?
3. Does every date contain 18 stocks in each downside-risk quintile?
4. Does every date contain 45 Higher Relative Risk and 45 Lower Relative Risk
   stocks?
5. Do the exact Notebook 09 rules reproduce the historical risk results reported
   in Notebook 08?
6. Do the highest-risk stocks experience deeper worst-path returns and more
   5% and 10% downside events?
7. Does the Attractive Risk-Reward class experience lower downside risk than the
   Unfavourable Risk-Reward class?
8. Are all rankings calculated independently within each date without
   look-ahead leakage?

## Reference Results from Notebook 08

For the cleaner 2025 onward period:

### Attractive Minus Unfavourable

| Metric | Expected difference |
|---|---:|
| Future return | −0.02 percentage points |
| Worst-path return | +1.84 percentage points |
| 5% downside-event rate | −13.70 percentage points |
| 10% downside-event rate | −9.60 percentage points |

### Highest-Risk Quintile Minus Lowest-Risk Quintile

| Metric | Expected difference |
|---|---:|
| Worst-path return | −1.45 percentage points |
| 5% downside-event rate | +13.58 percentage points |
| 10% downside-event rate | +8.33 percentage points |

Small differences in bootstrap confidence intervals are acceptable when caused
by random resampling, but the underlying monthly group results should closely
match Notebook 08.

## Interpretation Boundary

This notebook verifies downside-risk screening performance.

It does not attempt to establish:

- Guaranteed returns
- Exact calibrated probabilities
- Buy or sell recommendations
- A market-beating trading strategy
- Performance after transaction costs, slippage, or portfolio constraints

### Imports, independent paths, and verification configuration.

In [1]:
# Imports, paths, and verification configuration

from pathlib import Path
from datetime import datetime, timezone
import json
import math

import joblib
import numpy as np
import pandas as pd
from IPython.display import display


# ---------------------------------------------------------
# Resolve project root
# ---------------------------------------------------------

CURRENT_DIR = Path.cwd().resolve()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR


# ---------------------------------------------------------
# Independent source inputs
# ---------------------------------------------------------

FEATURE_DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "interim"
    / "targets"
    / "stock_features_with_targets_v1.parquet"
)

OUTPERFORM_MODEL_PATH = (
    PROJECT_ROOT
    / "models"
    / "best_random_forest_outperform_nifty50_20d_v1.joblib"
)

DOWNSIDE_MODEL_PATH = (
    PROJECT_ROOT
    / "models"
    / "random_forest_downside_10pct_20d_v1.joblib"
)


# Reference documents created by Notebooks 08 and 09.
# These will be used only to compare results, not to generate them.
NOTEBOOK_08_METADATA_PATH = (
    PROJECT_ROOT
    / "reports"
    / "historical_signal_backtest"
    / "historical_signal_backtest_metadata_v1.json"
)

NOTEBOOK_09_METADATA_PATH = (
    PROJECT_ROOT
    / "reports"
    / "rank_based_prediction_snapshot"
    / "rank_based_prediction_snapshot_metadata_2026-07-14_v2.json"
)


# Separate verification-report directory.
OUTPUT_DIR = (
    PROJECT_ROOT
    / "reports"
    / "rank_based_snapshot_verification_backtest"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ---------------------------------------------------------
# Historical verification configuration
# ---------------------------------------------------------

BACKTEST_START_DATE = pd.Timestamp("2024-01-01")
CLEAN_TEST_START_DATE = pd.Timestamp("2025-01-01")

MIN_STOCKS_PER_REBALANCE = 80
EXPECTED_STOCKS_PER_REBALANCE = 90

BOOTSTRAP_ITERATIONS = 10_000
BOOTSTRAP_RANDOM_SEED = 42

NOTEBOOK_VERSION = "10"
VERIFICATION_VERSION = "v1"
CLASSIFICATION_METHOD = "rank_based_relative_downside_risk"


# Future outcomes required to evaluate each historical signal.
OUTCOME_COLUMNS = [
    "future_return_20d",
    "future_nifty50_return_20d",
    "future_excess_return_20d_vs_nifty50",
    "future_min_return_20d",
    "future_max_return_20d",
    "target_outperform_nifty50_20d",
    "target_downside_5pct_20d",
    "target_big_downside_10pct_20d",
]


# ---------------------------------------------------------
# Validate required paths
# ---------------------------------------------------------

required_paths = {
    "Feature dataset": FEATURE_DATA_PATH,
    "Outperform model": OUTPERFORM_MODEL_PATH,
    "Downside model": DOWNSIDE_MODEL_PATH,
    "Notebook 08 metadata": NOTEBOOK_08_METADATA_PATH,
    "Notebook 09 metadata": NOTEBOOK_09_METADATA_PATH,
}


print("Project root:")
print(PROJECT_ROOT)

print("\nRequired inputs:")
for label, path in required_paths.items():
    status = "FOUND" if path.exists() else "MISSING"
    print(f"{label:<22} [{status}] {path}")

print("\nVerification configuration:")
print("Backtest start:", BACKTEST_START_DATE.date())
print("Clean test start:", CLEAN_TEST_START_DATE.date())
print(
    "Minimum monthly coverage:",
    MIN_STOCKS_PER_REBALANCE,
)
print(
    "Expected stocks per complete date:",
    EXPECTED_STOCKS_PER_REBALANCE,
)
print("Bootstrap iterations:", BOOTSTRAP_ITERATIONS)
print("Bootstrap seed:", BOOTSTRAP_RANDOM_SEED)

print("\nVerification output directory:")
print(OUTPUT_DIR)


missing_paths = [
    str(path)
    for path in required_paths.values()
    if not path.exists()
]

if missing_paths:
    raise FileNotFoundError(
        "The following required files were not found:\n"
        + "\n".join(missing_paths)
    )

print("\nPath and configuration validation passed.")

Project root:
E:\Projects\marketguard-india

Required inputs:
Feature dataset        [FOUND] E:\Projects\marketguard-india\data\interim\targets\stock_features_with_targets_v1.parquet
Outperform model       [FOUND] E:\Projects\marketguard-india\models\best_random_forest_outperform_nifty50_20d_v1.joblib
Downside model         [FOUND] E:\Projects\marketguard-india\models\random_forest_downside_10pct_20d_v1.joblib
Notebook 08 metadata   [FOUND] E:\Projects\marketguard-india\reports\historical_signal_backtest\historical_signal_backtest_metadata_v1.json
Notebook 09 metadata   [FOUND] E:\Projects\marketguard-india\reports\rank_based_prediction_snapshot\rank_based_prediction_snapshot_metadata_2026-07-14_v2.json

Verification configuration:
Backtest start: 2024-01-01
Clean test start: 2025-01-01
Minimum monthly coverage: 80
Expected stocks per complete date: 90
Bootstrap iterations: 10000
Bootstrap seed: 42

Verification output directory:
E:\Projects\marketguard-india\reports\rank_based_snapsho

### Load the dataset, models, and reference metadata.

In [2]:
#  Load dataset, models, and reference metadata

feature_data = pd.read_parquet(FEATURE_DATA_PATH)
outperform_model = joblib.load(OUTPERFORM_MODEL_PATH)
downside_model = joblib.load(DOWNSIDE_MODEL_PATH)

with NOTEBOOK_08_METADATA_PATH.open("r", encoding="utf-8") as file:
    notebook_08_metadata = json.load(file)

with NOTEBOOK_09_METADATA_PATH.open("r", encoding="utf-8") as file:
    notebook_09_metadata = json.load(file)


def extract_feature_names(fitted_model) -> list[str]:
    """Extract the ordered feature names stored in a fitted sklearn model."""

    if hasattr(fitted_model, "feature_names_in_"):
        return list(fitted_model.feature_names_in_)

    if hasattr(fitted_model, "named_steps"):
        for step in fitted_model.named_steps.values():
            if hasattr(step, "feature_names_in_"):
                return list(step.feature_names_in_)

    raise AttributeError(
        "Could not find feature_names_in_ in the fitted model or pipeline."
    )


outperform_features = extract_feature_names(outperform_model)
downside_features = extract_feature_names(downside_model)
same_feature_order = outperform_features == downside_features

if not same_feature_order:
    raise ValueError(
        "The outperform and downside models do not use the same ordered features."
    )

MODEL_FEATURES = outperform_features


required_control_columns = [
    "date",
    "yf_ticker",
    "model_ready_v1",
    "target_ready_v1_clean_20d",
]

required_dataset_columns = required_control_columns + MODEL_FEATURES + OUTCOME_COLUMNS

missing_required_columns = [
    column for column in required_dataset_columns
    if column not in feature_data.columns
]

if missing_required_columns:
    raise KeyError(
        "The historical dataset is missing required columns:\n"
        + "\n".join(missing_required_columns)
    )


feature_data["date"] = pd.to_datetime(feature_data["date"])

outperform_classes = np.asarray(outperform_model.classes_)
downside_classes = np.asarray(downside_model.classes_)

if 1 not in outperform_classes:
    raise ValueError("The outperform model does not contain target class 1.")

if 1 not in downside_classes:
    raise ValueError("The downside model does not contain target class 1.")


duplicate_stock_date_rows = int(
    feature_data.duplicated(subset=["date", "yf_ticker"]).sum()
)

if duplicate_stock_date_rows:
    raise ValueError(
        f"The dataset contains {duplicate_stock_date_rows} duplicate date/ticker rows."
    )


missing_model_feature_columns = [
    feature for feature in MODEL_FEATURES
    if feature not in feature_data.columns
]

missing_outcome_columns = [
    outcome for outcome in OUTCOME_COLUMNS
    if outcome not in feature_data.columns
]


print("Historical dataset:")
print("Shape:", feature_data.shape)
print("Date range:", feature_data["date"].min().date(), "to", feature_data["date"].max().date())
print("Unique stocks:", feature_data["yf_ticker"].nunique())
print("Duplicate date/ticker rows:", duplicate_stock_date_rows)

print("\nLoaded models:")
print("Outperform model:", type(outperform_model).__name__)
print("Downside model:", type(downside_model).__name__)
print("Outperform classes:", outperform_classes.tolist())
print("Downside classes:", downside_classes.tolist())

print("\nModel features:")
print("Outperform feature count:", len(outperform_features))
print("Downside feature count:", len(downside_features))
print("Identical ordered feature list:", same_feature_order)
print("Missing model-feature columns:", len(missing_model_feature_columns))

print("\nOutcome validation:")
print("Required outcomes:", len(OUTCOME_COLUMNS))
print("Missing outcome columns:", len(missing_outcome_columns))

outcome_validation = pd.DataFrame({
    "outcome_column": OUTCOME_COLUMNS,
    "present": [column in feature_data.columns for column in OUTCOME_COLUMNS],
    "non_missing_rows": [
        int(feature_data[column].notna().sum())
        for column in OUTCOME_COLUMNS
    ],
})

display(outcome_validation)

print("\nReference metadata:")
print("Notebook 08 top-level keys:", sorted(notebook_08_metadata.keys()))
print("Notebook 09 snapshot version:", notebook_09_metadata.get("snapshot_version"))
print("Notebook 09 classification method:", notebook_09_metadata.get("classification_method"))


if notebook_09_metadata.get("classification_method") != CLASSIFICATION_METHOD:
    raise ValueError(
        "Notebook 09 metadata does not contain the expected classification method."
    )

metadata_feature_count = notebook_09_metadata.get("model_configuration", {}).get(
    "feature_count"
)

if metadata_feature_count != len(MODEL_FEATURES):
    raise ValueError(
        "Notebook 09 metadata feature count does not match the fitted models."
    )


print("\nDataset, model, outcome, and metadata validation passed.")

Historical dataset:
Shape: (357370, 200)
Date range: 2010-01-04 to 2026-07-14
Unique stocks: 100
Duplicate date/ticker rows: 0

Loaded models:
Outperform model: Pipeline
Downside model: Pipeline
Outperform classes: [0, 1]
Downside classes: [0, 1]

Model features:
Outperform feature count: 79
Downside feature count: 79
Identical ordered feature list: True
Missing model-feature columns: 0

Outcome validation:
Required outcomes: 8
Missing outcome columns: 0


,outcome_column,present,non_missing_rows
0,future_return_20d,True,355370
1,future_nifty50_return_20d,True,351392
2,future_excess_return_20d_vs_nifty50,True,351392
3,future_min_return_20d,True,355370
4,future_max_return_20d,True,355370
5,target_outperform_nifty50_20d,True,307453
6,target_downside_5pct_20d,True,355370
7,target_big_downside_10pct_20d,True,355370



Reference metadata:
Notebook 08 top-level keys: ['backtest_version', 'clean_test_period', 'final_interpretation', 'full_backtest_period', 'generated_at', 'rank_based_attractive_minus_unfavourable_2025plus', 'recommended_design']
Notebook 09 snapshot version: v2
Notebook 09 classification method: rank_based_relative_downside_risk

Dataset, model, outcome, and metadata validation passed.


### Recreate the monthly historical sample

In [3]:
# Recreate the monthly historical verification sample

historical_data = feature_data.loc[
    feature_data["date"] >= BACKTEST_START_DATE
].copy()

historical_data["model_ready_v1"] = (
    historical_data["model_ready_v1"].fillna(False).astype(bool)
)

historical_data["target_ready_v1_clean_20d"] = (
    historical_data["target_ready_v1_clean_20d"].fillna(False).astype(bool)
)

historical_data["complete_outcomes"] = (
    historical_data[OUTCOME_COLUMNS].notna().all(axis=1)
)

historical_data["historically_eligible"] = (
    historical_data["model_ready_v1"]
    & historical_data["target_ready_v1_clean_20d"]
    & historical_data["complete_outcomes"]
)


eligible_rows = historical_data.loc[
    historical_data["historically_eligible"]
].copy()


# Count eligible stocks on every available date.
daily_coverage = (
    eligible_rows.groupby("date")["yf_ticker"]
    .nunique()
    .rename("eligible_stocks")
    .reset_index()
)

daily_coverage["month"] = daily_coverage["date"].dt.to_period("M")


# Select the first date in each month with sufficient cross-sectional coverage.
qualifying_dates = daily_coverage.loc[
    daily_coverage["eligible_stocks"] >= MIN_STOCKS_PER_REBALANCE
].copy()

selected_rebalance_dates = (
    qualifying_dates.sort_values("date")
    .groupby("month", as_index=False)
    .first()
    .rename(columns={"date": "rebalance_date"})
)


verification_sample = eligible_rows.merge(
    selected_rebalance_dates[["month", "rebalance_date"]],
    left_on="date",
    right_on="rebalance_date",
    how="inner",
)

verification_sample = (
    verification_sample.drop(columns=["rebalance_date"])
    .sort_values(["date", "yf_ticker"])
    .reset_index(drop=True)
)


# Validate the selected monthly sample.
stocks_per_rebalance = (
    verification_sample.groupby("date")["yf_ticker"].nunique()
)

rebalance_date_count = verification_sample["date"].nunique()
verification_row_count = len(verification_sample)

incomplete_selected_dates = stocks_per_rebalance.loc[
    stocks_per_rebalance != EXPECTED_STOCKS_PER_REBALANCE
]

if not incomplete_selected_dates.empty:
    raise ValueError(
        "Some selected rebalance dates do not contain exactly "
        f"{EXPECTED_STOCKS_PER_REBALANCE} stocks:\n"
        f"{incomplete_selected_dates}"
    )

if verification_sample.duplicated(["date", "yf_ticker"]).any():
    raise ValueError("The verification sample contains duplicate stock/date rows.")

if rebalance_date_count != 30:
    raise ValueError(
        f"Expected 30 monthly rebalance dates, found {rebalance_date_count}."
    )

if verification_row_count != 2_700:
    raise ValueError(
        f"Expected 2,700 historical observations, found {verification_row_count}."
    )


clean_test_sample = verification_sample.loc[
    verification_sample["date"] >= CLEAN_TEST_START_DATE
].copy()

clean_test_date_count = clean_test_sample["date"].nunique()


print("Historical eligibility:")
print("Rows from 2024 onward:", len(historical_data))
print("Eligible rows with complete outcomes:", len(eligible_rows))
print("Available eligible dates:", eligible_rows["date"].nunique())

print("\nSelected monthly verification sample:")
print("Rebalance dates:", rebalance_date_count)
print("Rows:", verification_row_count)
print("Stocks per date:", int(stocks_per_rebalance.min()), "to", int(stocks_per_rebalance.max()))
print("Date range:", verification_sample["date"].min().date(), "to", verification_sample["date"].max().date())

print("\nCleaner 2025+ test period:")
print("Rebalance dates:", clean_test_date_count)
print("Rows:", len(clean_test_sample))
print("Date range:", clean_test_sample["date"].min().date(), "to", clean_test_sample["date"].max().date())

print("\nSelected rebalance dates:")
display(selected_rebalance_dates)

print("\nApril 2026 daily coverage:")
display(
    daily_coverage.loc[
        daily_coverage["month"] == pd.Period("2026-04", freq="M")
    ].head(10)
)

print("\nMonthly historical sample validation passed.")

Historical eligibility:
Rows from 2024 onward: 61336
Eligible rows with complete outcomes: 53640
Available eligible dates: 600

Selected monthly verification sample:
Rebalance dates: 30
Rows: 2700
Stocks per date: 90 to 90
Date range: 2024-01-02 to 2026-06-01

Cleaner 2025+ test period:
Rebalance dates: 18
Rows: 1620
Date range: 2025-01-01 to 2026-06-01

Selected rebalance dates:


,month,rebalance_date,eligible_stocks
0,2024-01,2024-01-02,90
1,2024-02,2024-02-01,90
2,2024-03,2024-03-01,90
3,2024-04,2024-04-01,90
4,2024-05,2024-05-02,90
5,2024-06,2024-06-03,90
6,2024-07,2024-07-01,90
7,2024-08,2024-08-01,90
8,2024-09,2024-09-02,90
9,2024-10,2024-10-01,90



April 2026 daily coverage:


,date,eligible_stocks,month
550,2026-04-01,3,2026-04
551,2026-04-02,90,2026-04
552,2026-04-06,90,2026-04
553,2026-04-07,90,2026-04
554,2026-04-08,90,2026-04
555,2026-04-09,90,2026-04
556,2026-04-10,90,2026-04
557,2026-04-13,90,2026-04
558,2026-04-15,90,2026-04
559,2026-04-16,90,2026-04



Monthly historical sample validation passed.



reproduced Notebook 08’s historical sample exactly:

        30 monthly dates
        2,700 observations
        90 stocks on every selected date
        18 clean test months from 2025 onward
        April 2026 correctly moved from April 1, with only 3 eligible stocks, to April 2, with 90

Now apply the two fixed models and Notebook 09’s exact classification rules.

### Score and classify every historical cross-section

In [4]:
#  Generate historical scores, ranks, and exact v2 classifications

historical_scored = verification_sample.copy()

missing_feature_values = int(
    historical_scored[MODEL_FEATURES].isna().sum().sum()
)

if missing_feature_values:
    raise ValueError(
        f"The historical sample contains {missing_feature_values} missing model-feature values."
    )


def predict_positive_probability(model, features: pd.DataFrame) -> np.ndarray:
    """Return predict_proba values for fitted target class 1."""

    class_locations = np.flatnonzero(np.asarray(model.classes_) == 1)

    if len(class_locations) != 1:
        raise ValueError(f"Expected one positive class. Found: {model.classes_.tolist()}")

    return model.predict_proba(features)[:, int(class_locations[0])]


X_historical = historical_scored[MODEL_FEATURES]

historical_scored["outperform_probability"] = predict_positive_probability(
    outperform_model,
    X_historical,
)

historical_scored["downside_probability"] = predict_positive_probability(
    downside_model,
    X_historical,
)


# Rank stocks independently within each rebalance date.
historical_scored["outperform_rank"] = (
    historical_scored.groupby("date")["outperform_probability"]
    .rank(method="first", ascending=False)
    .astype("Int64")
)

historical_scored["downside_risk_rank"] = (
    historical_scored.groupby("date")["downside_probability"]
    .rank(method="first", ascending=False)
    .astype("Int64")
)


# Retain Notebook 07 fixed probability bands for later comparison.
historical_scored["downside_risk_band"] = pd.cut(
    historical_scored["downside_probability"],
    bins=[-np.inf, 0.40, 0.47, 0.51, np.inf],
    labels=["Low Risk", "Watch Risk", "High Risk", "Very High Risk"],
    right=False,
).astype("string")


# Exact Notebook 09 opportunity tiers.
historical_scored["opportunity_tier"] = np.select(
    [
        historical_scored["outperform_rank"] <= 18,
        historical_scored["outperform_rank"] <= 45,
    ],
    [
        "High Opportunity",
        "Moderate Opportunity",
    ],
    default="Low Opportunity",
)


# Exact Notebook 09 detailed risk quintiles.
historical_scored["downside_risk_quintile"] = np.select(
    [
        historical_scored["downside_risk_rank"] <= 18,
        historical_scored["downside_risk_rank"] <= 36,
        historical_scored["downside_risk_rank"] <= 54,
        historical_scored["downside_risk_rank"] <= 72,
    ],
    [
        "Q1 — Highest Risk",
        "Q2 — High Risk",
        "Q3 — Medium Risk",
        "Q4 — Lower Risk",
    ],
    default="Q5 — Lowest Risk",
)


# Exact Notebook 09 broad 50/50 risk split.
historical_scored["relative_downside_risk"] = np.where(
    historical_scored["downside_risk_rank"] <= 45,
    "Higher Relative Risk",
    "Lower Relative Risk",
)


# Exact Notebook 09 combined classification.
historical_scored["opportunity_risk_class"] = np.select(
    [
        (
            historical_scored["opportunity_tier"].eq("High Opportunity")
            & historical_scored["relative_downside_risk"].eq("Lower Relative Risk")
        ),
        (
            historical_scored["opportunity_tier"].eq("High Opportunity")
            & historical_scored["relative_downside_risk"].eq("Higher Relative Risk")
        ),
        (
            historical_scored["opportunity_tier"].eq("Moderate Opportunity")
            & historical_scored["relative_downside_risk"].eq("Lower Relative Risk")
        ),
        (
            historical_scored["opportunity_tier"].eq("Moderate Opportunity")
            & historical_scored["relative_downside_risk"].eq("Higher Relative Risk")
        ),
        (
            historical_scored["opportunity_tier"].eq("Low Opportunity")
            & historical_scored["relative_downside_risk"].eq("Lower Relative Risk")
        ),
    ],
    [
        "Attractive Risk-Reward",
        "High Opportunity / High Risk",
        "Balanced Opportunity",
        "Caution",
        "Low Opportunity / Lower Risk",
    ],
    default="Unfavourable Risk-Reward",
)


# Validate ranks and classification counts independently on every date.
date_validation = historical_scored.groupby("date").agg(
    stock_count=("yf_ticker", "nunique"),
    opportunity_rank_min=("outperform_rank", "min"),
    opportunity_rank_max=("outperform_rank", "max"),
    opportunity_rank_unique=("outperform_rank", "nunique"),
    risk_rank_min=("downside_risk_rank", "min"),
    risk_rank_max=("downside_risk_rank", "max"),
    risk_rank_unique=("downside_risk_rank", "nunique"),
)

if not date_validation["stock_count"].eq(90).all():
    raise ValueError("At least one date does not contain exactly 90 stocks.")

if not date_validation["opportunity_rank_unique"].eq(90).all():
    raise ValueError("Opportunity ranks are not unique within every date.")

if not date_validation["risk_rank_unique"].eq(90).all():
    raise ValueError("Downside-risk ranks are not unique within every date.")

if not date_validation["opportunity_rank_min"].eq(1).all():
    raise ValueError("Opportunity ranks do not begin at 1 on every date.")

if not date_validation["opportunity_rank_max"].eq(90).all():
    raise ValueError("Opportunity ranks do not end at 90 on every date.")

if not date_validation["risk_rank_min"].eq(1).all():
    raise ValueError("Downside-risk ranks do not begin at 1 on every date.")

if not date_validation["risk_rank_max"].eq(90).all():
    raise ValueError("Downside-risk ranks do not end at 90 on every date.")


opportunity_counts_by_date = pd.crosstab(
    historical_scored["date"],
    historical_scored["opportunity_tier"],
)

expected_opportunity_counts = {
    "High Opportunity": 18,
    "Moderate Opportunity": 27,
    "Low Opportunity": 45,
}

for label, expected_count in expected_opportunity_counts.items():
    if not opportunity_counts_by_date[label].eq(expected_count).all():
        raise ValueError(f"{label} does not contain {expected_count} stocks on every date.")


quintile_counts_by_date = pd.crosstab(
    historical_scored["date"],
    historical_scored["downside_risk_quintile"],
)

for label in [
    "Q1 — Highest Risk",
    "Q2 — High Risk",
    "Q3 — Medium Risk",
    "Q4 — Lower Risk",
    "Q5 — Lowest Risk",
]:
    if not quintile_counts_by_date[label].eq(18).all():
        raise ValueError(f"{label} does not contain 18 stocks on every date.")


relative_risk_counts_by_date = pd.crosstab(
    historical_scored["date"],
    historical_scored["relative_downside_risk"],
)

for label in ["Higher Relative Risk", "Lower Relative Risk"]:
    if not relative_risk_counts_by_date[label].eq(45).all():
        raise ValueError(f"{label} does not contain 45 stocks on every date.")


print("Historical scoring:")
print("Rows scored:", len(historical_scored))
print("Rebalance dates:", historical_scored["date"].nunique())
print("Missing model-feature values:", missing_feature_values)
print(
    "Outperform score range:",
    f"{historical_scored['outperform_probability'].min():.6f}",
    "to",
    f"{historical_scored['outperform_probability'].max():.6f}",
)
print(
    "Downside score range:",
    f"{historical_scored['downside_probability'].min():.6f}",
    "to",
    f"{historical_scored['downside_probability'].max():.6f}",
)

print("\nOpportunity counts on every date:")
display(opportunity_counts_by_date.drop_duplicates())

print("\nRisk-quintile counts on every date:")
display(quintile_counts_by_date.drop_duplicates())

print("\nBroad relative-risk counts on every date:")
display(relative_risk_counts_by_date.drop_duplicates())

print("\nOverall combined-class counts:")
display(
    historical_scored["opportunity_risk_class"]
    .value_counts()
    .rename_axis("opportunity_risk_class")
    .reset_index(name="stock_count")
)

print("\nHistorical scoring and exact v2 classification validation passed.")

Historical scoring:
Rows scored: 2700
Rebalance dates: 30
Missing model-feature values: 0
Outperform score range: 0.327965 to 0.675033
Downside score range: 0.085977 to 0.759843

Opportunity counts on every date:


opportunity_tier,High Opportunity,Low Opportunity,Moderate Opportunity
date,,,
2024-01-02,18,45,27



Risk-quintile counts on every date:


downside_risk_quintile,Q1 — Highest Risk,Q2 — High Risk,Q3 — Medium Risk,Q4 — Lower Risk,Q5 — Lowest Risk
date,,,,,
2024-01-02,18,18,18,18,18



Broad relative-risk counts on every date:


relative_downside_risk,Higher Relative Risk,Lower Relative Risk
date,,
2024-01-02,45,45



Overall combined-class counts:


,opportunity_risk_class,stock_count
0,Unfavourable Risk-Reward,742
1,Low Opportunity / Lower Risk,608
2,Balanced Opportunity,446
3,Caution,364
4,Attractive Risk-Reward,296
5,High Opportunity / High Risk,244



Historical scoring and exact v2 classification validation passed.


The exact Notebook 09 rules were applied consistently across all 30 historical dates:

        2,700 rows scored
        90 unique ranks per date
        18 stocks in every risk quintile
        45 stocks in each broad risk group
        No missing model inputs

The displayed per-date tables show only one row because .drop_duplicates() confirmed that every rebalance date had the same counts.

Now calculate the actual historical outcomes, using each month equally rather than allowing larger classes to dominate.

### Calculate 2025+ class and quintile performance

In [5]:
# Calculate clean-period historical performance

COMBINED_CLASS_ORDER = [
    "Attractive Risk-Reward",
    "High Opportunity / High Risk",
    "Balanced Opportunity",
    "Caution",
    "Low Opportunity / Lower Risk",
    "Unfavourable Risk-Reward",
]

RISK_QUINTILE_ORDER = [
    "Q1 — Highest Risk",
    "Q2 — High Risk",
    "Q3 — Medium Risk",
    "Q4 — Lower Risk",
    "Q5 — Lowest Risk",
]


def calculate_monthly_group_performance(
    data: pd.DataFrame,
    group_column: str,
) -> pd.DataFrame:
    """Create equal-weight performance results for each group and month."""

    return (
        data.groupby(["date", group_column])
        .agg(
            stock_count=("yf_ticker", "nunique"),
            future_return=("future_return_20d", "mean"),
            excess_return=("future_excess_return_20d_vs_nifty50", "mean"),
            worst_path_return=("future_min_return_20d", "mean"),
            outperform_rate=("target_outperform_nifty50_20d", "mean"),
            downside_5pct_rate=("target_downside_5pct_20d", "mean"),
            downside_10pct_rate=("target_big_downside_10pct_20d", "mean"),
        )
        .reset_index()
    )


def summarize_monthly_performance(
    monthly_data: pd.DataFrame,
    group_column: str,
    group_order: list[str],
) -> pd.DataFrame:
    """Average monthly group results so every month has equal weight."""

    summary = (
        monthly_data.groupby(group_column)
        .agg(
            months=("date", "nunique"),
            average_stocks_per_month=("stock_count", "mean"),
            average_future_return=("future_return", "mean"),
            average_excess_return=("excess_return", "mean"),
            average_worst_path_return=("worst_path_return", "mean"),
            outperform_rate=("outperform_rate", "mean"),
            downside_5pct_rate=("downside_5pct_rate", "mean"),
            downside_10pct_rate=("downside_10pct_rate", "mean"),
        )
        .reindex(group_order)
        .reset_index()
    )

    percentage_columns = [
        "average_future_return",
        "average_excess_return",
        "average_worst_path_return",
        "outperform_rate",
        "downside_5pct_rate",
        "downside_10pct_rate",
    ]

    summary[percentage_columns] = summary[percentage_columns] * 100
    return summary


def calculate_paired_spread(
    monthly_data: pd.DataFrame,
    group_column: str,
    first_group: str,
    second_group: str,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Calculate first-group minus second-group differences on common months."""

    metric_columns = [
        "future_return",
        "excess_return",
        "worst_path_return",
        "outperform_rate",
        "downside_5pct_rate",
        "downside_10pct_rate",
    ]

    first = (
        monthly_data.loc[monthly_data[group_column].eq(first_group), ["date"] + metric_columns]
        .set_index("date")
    )

    second = (
        monthly_data.loc[monthly_data[group_column].eq(second_group), ["date"] + metric_columns]
        .set_index("date")
    )

    common_dates = first.index.intersection(second.index)
    monthly_difference = first.loc[common_dates] - second.loc[common_dates]
    monthly_difference = monthly_difference.reset_index()

    spread_summary = pd.DataFrame({
        "metric": metric_columns,
        "difference_percentage_points": [
            monthly_difference[column].mean() * 100
            for column in metric_columns
        ],
    })

    return monthly_difference, spread_summary


clean_scored = historical_scored.loc[
    historical_scored["date"] >= CLEAN_TEST_START_DATE
].copy()


# Combined-class performance
monthly_class_performance = calculate_monthly_group_performance(
    clean_scored,
    "opportunity_risk_class",
)

class_summary_2025plus = summarize_monthly_performance(
    monthly_class_performance,
    "opportunity_risk_class",
    COMBINED_CLASS_ORDER,
)


# Detailed risk-quintile performance
monthly_quintile_performance = calculate_monthly_group_performance(
    clean_scored,
    "downside_risk_quintile",
)

quintile_summary_2025plus = summarize_monthly_performance(
    monthly_quintile_performance,
    "downside_risk_quintile",
    RISK_QUINTILE_ORDER,
)


# Attractive minus Unfavourable
attractive_unfavourable_monthly_spread, attractive_unfavourable_spread = (
    calculate_paired_spread(
        monthly_class_performance,
        "opportunity_risk_class",
        "Attractive Risk-Reward",
        "Unfavourable Risk-Reward",
    )
)


# Highest-risk quintile minus lowest-risk quintile
highest_lowest_risk_monthly_spread, highest_lowest_risk_spread = (
    calculate_paired_spread(
        monthly_quintile_performance,
        "downside_risk_quintile",
        "Q1 — Highest Risk",
        "Q5 — Lowest Risk",
    )
)


# Validate that every group appears in all 18 clean test months.
if not class_summary_2025plus["months"].eq(18).all():
    raise ValueError("Not every combined class appears in all 18 test months.")

if not quintile_summary_2025plus["months"].eq(18).all():
    raise ValueError("Not every risk quintile appears in all 18 test months.")

if len(attractive_unfavourable_monthly_spread) != 18:
    raise ValueError("Attractive and Unfavourable do not share exactly 18 months.")

if len(highest_lowest_risk_monthly_spread) != 18:
    raise ValueError("Q1 and Q5 do not share exactly 18 months.")


print("Clean test period:")
print("Dates:", clean_scored["date"].nunique())
print("Rows:", len(clean_scored))
print("Date range:", clean_scored["date"].min().date(), "to", clean_scored["date"].max().date())

print("\nRank-based combined-class performance, 2025+:")
display(class_summary_2025plus.round(2))

print("\nDownside-risk quintile performance, 2025+:")
display(quintile_summary_2025plus.round(2))

print("\nAttractive minus Unfavourable point estimates:")
display(attractive_unfavourable_spread.round(2))

print("\nQ1 Highest Risk minus Q5 Lowest Risk point estimates:")
display(highest_lowest_risk_spread.round(2))

print("\nClean-period performance calculation passed.")

Clean test period:
Dates: 18
Rows: 1620
Date range: 2025-01-01 to 2026-06-01

Rank-based combined-class performance, 2025+:


,opportunity_risk_class,months,average_stocks_per_month,average_future_return,average_excess_return,average_worst_path_return,outperform_rate,downside_5pct_rate,downside_10pct_rate
0,Attractive Risk-Reward,18,10.39,0.88,0.79,-3.04,54.43,25.02,6.45
1,High Opportunity / High Risk,18,7.61,1.22,1.14,-4.98,56.49,44.91,15.13
2,Balanced Opportunity,18,15.83,0.48,0.40,-3.52,52.39,28.37,10.94
3,Caution,18,11.17,2.26,2.17,-3.74,58.07,34.53,13.94
4,Low Opportunity / Lower Risk,18,18.78,0.60,0.51,-3.74,54.41,28.98,11.91
5,Unfavourable Risk-Reward,18,26.22,0.90,0.81,-4.88,50.7,38.72,16.05



Downside-risk quintile performance, 2025+:


,downside_risk_quintile,months,average_stocks_per_month,average_future_return,average_excess_return,average_worst_path_return,outperform_rate,downside_5pct_rate,downside_10pct_rate
0,Q1 — Highest Risk,18,18.0,1.65,1.57,-4.96,54.94,40.74,18.52
1,Q2 — High Risk,18,18.0,1.51,1.42,-4.22,55.56,35.49,12.35
2,Q3 — Medium Risk,18,18.0,0.54,0.45,-4.35,50.31,34.57,14.2
3,Q4 — Lower Risk,18,18.0,0.53,0.44,-3.50,54.63,27.78,9.57
4,Q5 — Lowest Risk,18,18.0,0.32,0.24,-3.51,51.54,27.16,10.19



Attractive minus Unfavourable point estimates:


,metric,difference_percentage_points
0,future_return,-0.02
1,excess_return,-0.02
2,worst_path_return,1.84
3,outperform_rate,3.73
4,downside_5pct_rate,-13.70
5,downside_10pct_rate,-9.60



Q1 Highest Risk minus Q5 Lowest Risk point estimates:


,metric,difference_percentage_points
0,future_return,1.33
1,excess_return,1.34
2,worst_path_return,-1.45
3,outperform_rate,3.40
4,downside_5pct_rate,13.58
5,downside_10pct_rate,8.33



Clean-period performance calculation passed.


## 2025+ Rank-Based Backtest Summary

The cleaner evaluation period covers **18 monthly rebalance dates** from
January 2025 through June 2026.

Each month contains:

- 90 eligible stocks
- 18 High Opportunity stocks
- 27 Moderate Opportunity stocks
- 45 Low Opportunity stocks
- 18 stocks in each downside-risk quintile
- 45 Higher Relative Risk stocks
- 45 Lower Relative Risk stocks

Performance is first calculated within each month and then averaged across the
18 months. This gives every rebalance month equal weight.

---

## Metric Interpretation

| Metric | Meaning |
|---|---|
| Average future return | Final stock return after 20 trading days |
| Average excess return | Stock return minus Nifty 50 return over the same period |
| Average worst-path return | Lowest return reached at any point during the 20-day period |
| Outperform rate | Percentage of stocks that finished ahead of Nifty 50 |
| 5% downside-event rate | Percentage that fell at least 5% during the 20-day path |
| 10% downside-event rate | Percentage that fell at least 10% during the 20-day path |

Final return and worst-path return measure different outcomes. A stock may fall
sharply during the holding period and still recover to finish with a positive
return.

---

## Rank-Based Combined-Class Performance

| Opportunity-risk class | Avg. stocks/month | Future return | Excess return | Worst path | Outperform rate | 5% downside rate | 10% downside rate |
|---|---:|---:|---:|---:|---:|---:|---:|
| Attractive Risk-Reward | 10.39 | 0.88% | 0.79% | −3.04% | 54.43% | 25.02% | 6.45% |
| High Opportunity / High Risk | 7.61 | 1.22% | 1.14% | −4.98% | 56.49% | 44.91% | 15.13% |
| Balanced Opportunity | 15.83 | 0.48% | 0.40% | −3.52% | 52.39% | 28.37% | 10.94% |
| Caution | 11.17 | 2.26% | 2.17% | −3.74% | 58.07% | 34.53% | 13.94% |
| Low Opportunity / Lower Risk | 18.78 | 0.60% | 0.51% | −3.74% | 54.41% | 28.98% | 11.91% |
| Unfavourable Risk-Reward | 26.22 | 0.90% | 0.81% | −4.88% | 50.70% | 38.72% | 16.05% |

### Main Observations

- **Attractive Risk-Reward** had the shallowest average worst-path return and
  the lowest 5% and 10% downside-event rates.
- **High Opportunity / High Risk** produced somewhat higher final returns than
  Attractive stocks, but experienced much deeper drawdowns and more frequent
  downside events.
- **Balanced Opportunity** had a relatively controlled downside profile but
  weaker final returns.
- **Caution** produced the highest observed final return and outperform rate,
  but also had materially higher downside-event rates than Balanced
  Opportunity.
- **Unfavourable Risk-Reward** had almost the same final return as Attractive
  Risk-Reward, but experienced significantly worse drawdown behaviour.

The classes therefore separate **downside exposure** more clearly than they
separate final returns.

---

## Downside-Risk Quintile Performance

| Downside-risk quintile | Stocks/month | Future return | Excess return | Worst path | Outperform rate | 5% downside rate | 10% downside rate |
|---|---:|---:|---:|---:|---:|---:|---:|
| Q1 — Highest Risk | 18 | 1.65% | 1.57% | −4.96% | 54.94% | 40.74% | 18.52% |
| Q2 — High Risk | 18 | 1.51% | 1.42% | −4.22% | 55.56% | 35.49% | 12.35% |
| Q3 — Medium Risk | 18 | 0.54% | 0.45% | −4.35% | 50.31% | 34.57% | 14.20% |
| Q4 — Lower Risk | 18 | 0.53% | 0.44% | −3.50% | 54.63% | 27.78% | 9.57% |
| Q5 — Lowest Risk | 18 | 0.32% | 0.24% | −3.51% | 51.54% | 27.16% | 10.19% |

### Main Observations

- Q1 Highest Risk had the deepest average worst-path return.
- Q1 also had the highest 5% and 10% downside-event rates.
- Q5 Lowest Risk had a substantially safer historical path than Q1.
- Final returns were higher in the riskier quintiles, showing that the downside
  model measures drawdown exposure rather than guaranteed terminal losses.
- The middle quintiles were not perfectly ordered, which is reasonable given
  the small 18-month test sample and noisy stock outcomes.
- The clearest separation appears between the extreme groups, Q1 and Q5.

---

## Attractive Versus Unfavourable

The paired comparison subtracts the Unfavourable result from the Attractive
result on the same monthly rebalance dates.

| Metric | Attractive minus Unfavourable |
|---|---:|
| Future return | −0.02 percentage points |
| Excess return | −0.02 percentage points |
| Worst-path return | +1.84 percentage points |
| Outperform rate | +3.73 percentage points |
| 5% downside-event rate | −13.70 percentage points |
| 10% downside-event rate | −9.60 percentage points |

### Interpretation

- Attractive and Unfavourable stocks produced almost identical final returns.
- Attractive stocks had drawdowns that were, on average, **1.84 percentage
  points shallower**.
- Attractive stocks experienced **13.70 percentage points fewer 5% downside
  events**.
- Attractive stocks experienced **9.60 percentage points fewer 10% downside
  events**.
- The combined class therefore appears useful primarily for risk screening,
  rather than for identifying a reliable return advantage.

---

## Highest-Risk Versus Lowest-Risk Quintiles

The comparison subtracts Q5 Lowest Risk from Q1 Highest Risk.

| Metric | Q1 minus Q5 |
|---|---:|
| Future return | +1.33 percentage points |
| Excess return | +1.34 percentage points |
| Worst-path return | −1.45 percentage points |
| Outperform rate | +3.40 percentage points |
| 5% downside-event rate | +13.58 percentage points |
| 10% downside-event rate | +8.33 percentage points |

### Interpretation

- Q1 stocks finished with higher average returns, but experienced materially
  worse interim drawdowns.
- Q1 drawdowns were **1.45 percentage points deeper** than Q5 drawdowns.
- Q1 experienced **13.58 percentage points more 5% downside events**.
- Q1 experienced **8.33 percentage points more 10% downside events**.
- This supports the downside model as a relative risk-ranking signal.

---

## Current Conclusion

The exact classification logic implemented in Notebook 09 reproduces the
historical point estimates from Notebook 08.

The current evidence indicates that:

- The downside model contains useful information about relative drawdown risk.
- The rank-based combined classes provide meaningful risk separation.
- Attractive Risk-Reward stocks historically experienced safer return paths
  than Unfavourable Risk-Reward stocks.
- The system has not demonstrated a reliable final-return or alpha advantage.
- Higher-risk stocks may still finish with strong returns after experiencing
  deeper interim losses.

These results support MarketGuard as a **risk-intelligence and stock-screening
framework**, not as a proven buy/sell or market-beating strategy.

Bootstrap confidence intervals are required next to determine which observed
differences were stable across the 18 monthly periods and which may have resulted
from sampling variation.

### Paired bootstrap confidence intervals

In [6]:
# Calculate paired bootstrap confidence intervals

METRIC_LABELS = {
    "future_return": "Future return",
    "excess_return": "Excess return",
    "worst_path_return": "Worst-path return",
    "outperform_rate": "Outperform rate",
    "downside_5pct_rate": "5% downside-event rate",
    "downside_10pct_rate": "10% downside-event rate",
}


def bootstrap_mean_ci(
    values: pd.Series,
    iterations: int = BOOTSTRAP_ITERATIONS,
    seed: int = BOOTSTRAP_RANDOM_SEED,
    confidence_level: float = 0.95,
) -> tuple[float, float, float]:
    """Bootstrap the mean paired monthly difference."""

    array = pd.to_numeric(values, errors="coerce").dropna().to_numpy(dtype=float)

    if len(array) < 2:
        raise ValueError("At least two paired observations are required.")

    rng = np.random.default_rng(seed)
    sample_indices = rng.integers(0, len(array), size=(iterations, len(array)))
    bootstrap_means = array[sample_indices].mean(axis=1)

    alpha = 1 - confidence_level
    lower = np.quantile(bootstrap_means, alpha / 2)
    upper = np.quantile(bootstrap_means, 1 - alpha / 2)

    return array.mean() * 100, lower * 100, upper * 100


def create_bootstrap_summary(monthly_spread: pd.DataFrame) -> pd.DataFrame:
    """Create bootstrap results for every paired performance metric."""

    records = []

    for metric, label in METRIC_LABELS.items():
        mean, lower, upper = bootstrap_mean_ci(monthly_spread[metric])

        records.append({
            "metric": metric,
            "metric_label": label,
            "common_months": int(monthly_spread[metric].notna().sum()),
            "difference_percentage_points": mean,
            "ci_95_lower": lower,
            "ci_95_upper": upper,
            "ci_excludes_zero": bool(lower > 0 or upper < 0),
        })

    return pd.DataFrame(records)


attractive_unfavourable_bootstrap = create_bootstrap_summary(
    attractive_unfavourable_monthly_spread
)

highest_lowest_risk_bootstrap = create_bootstrap_summary(
    highest_lowest_risk_monthly_spread
)


# Rounded Notebook 08 reference results.
combined_reference = pd.DataFrame([
    {"metric": "future_return", "expected_difference": -0.02, "expected_lower": -2.25, "expected_upper": 1.99},
    {"metric": "excess_return", "expected_difference": -0.02, "expected_lower": -2.26, "expected_upper": 1.99},
    {"metric": "worst_path_return", "expected_difference": 1.84, "expected_lower": 0.73, "expected_upper": 3.14},
    {"metric": "outperform_rate", "expected_difference": 3.73, "expected_lower": -10.33, "expected_upper": 17.83},
    {"metric": "downside_5pct_rate", "expected_difference": -13.70, "expected_lower": -24.59, "expected_upper": -3.98},
    {"metric": "downside_10pct_rate", "expected_difference": -9.60, "expected_lower": -19.70, "expected_upper": -1.99},
])

quintile_reference = pd.DataFrame([
    {"metric": "future_return", "expected_difference": 1.33, "expected_lower": -0.73, "expected_upper": 3.46},
    {"metric": "excess_return", "expected_difference": 1.34, "expected_lower": -0.72, "expected_upper": 3.47},
    {"metric": "worst_path_return", "expected_difference": -1.45, "expected_lower": -2.55, "expected_upper": -0.39},
    {"metric": "downside_5pct_rate", "expected_difference": 13.58, "expected_lower": 6.17, "expected_upper": 22.53},
    {"metric": "downside_10pct_rate", "expected_difference": 8.33, "expected_lower": 1.54, "expected_upper": 16.67},
])


def compare_with_reference(
    calculated: pd.DataFrame,
    reference: pd.DataFrame,
    tolerance: float = 0.10,
) -> pd.DataFrame:
    """Compare calculated bootstrap results with rounded Notebook 08 values."""

    comparison = reference.merge(calculated, on="metric", how="left")

    comparison["difference_delta"] = (
        comparison["difference_percentage_points"] - comparison["expected_difference"]
    )

    comparison["lower_delta"] = comparison["ci_95_lower"] - comparison["expected_lower"]
    comparison["upper_delta"] = comparison["ci_95_upper"] - comparison["expected_upper"]

    comparison["matches_reference"] = (
        comparison["difference_delta"].abs().le(tolerance)
        & comparison["lower_delta"].abs().le(tolerance)
        & comparison["upper_delta"].abs().le(tolerance)
    )

    return comparison


combined_reference_check = compare_with_reference(
    attractive_unfavourable_bootstrap,
    combined_reference,
)

quintile_reference_check = compare_with_reference(
    highest_lowest_risk_bootstrap,
    quintile_reference,
)


if not combined_reference_check["matches_reference"].all():
    raise ValueError(
        "Attractive-versus-Unfavourable bootstrap results do not reproduce "
        "Notebook 08 within tolerance."
    )

if not quintile_reference_check["matches_reference"].all():
    raise ValueError(
        "Highest-versus-lowest risk bootstrap results do not reproduce "
        "Notebook 08 within tolerance."
    )


print("Bootstrap configuration:")
print("Iterations:", BOOTSTRAP_ITERATIONS)
print("Random seed:", BOOTSTRAP_RANDOM_SEED)
print("Paired months:", len(attractive_unfavourable_monthly_spread))

print("\nAttractive minus Unfavourable bootstrap results:")
display(attractive_unfavourable_bootstrap.round(2))

print("\nQ1 Highest Risk minus Q5 Lowest Risk bootstrap results:")
display(highest_lowest_risk_bootstrap.round(2))

print("\nAttractive-minus-Unfavourable reference comparison:")
display(
    combined_reference_check[[
        "metric",
        "difference_delta",
        "lower_delta",
        "upper_delta",
        "matches_reference",
    ]].round(4)
)

print("\nQ1-minus-Q5 reference comparison:")
display(
    quintile_reference_check[[
        "metric",
        "difference_delta",
        "lower_delta",
        "upper_delta",
        "matches_reference",
    ]].round(4)
)

print("\nPaired bootstrap and Notebook 08 reference validation passed.")

Bootstrap configuration:
Iterations: 10000
Random seed: 42
Paired months: 18

Attractive minus Unfavourable bootstrap results:


,metric,metric_label,common_months,difference_percentage_points,ci_95_lower,ci_95_upper,ci_excludes_zero
0,future_return,Future return,18,-0.02,-2.25,1.99,False
1,excess_return,Excess return,18,-0.02,-2.26,1.99,False
2,worst_path_return,Worst-path return,18,1.84,0.73,3.14,True
3,outperform_rate,Outperform rate,18,3.73,-10.33,17.83,False
4,downside_5pct_rate,5% downside-event rate,18,-13.70,-24.59,-3.98,True
5,downside_10pct_rate,10% downside-event rate,18,-9.60,-19.70,-1.99,True



Q1 Highest Risk minus Q5 Lowest Risk bootstrap results:


,metric,metric_label,common_months,difference_percentage_points,ci_95_lower,ci_95_upper,ci_excludes_zero
0,future_return,Future return,18,1.33,-0.73,3.46,False
1,excess_return,Excess return,18,1.34,-0.72,3.47,False
2,worst_path_return,Worst-path return,18,-1.45,-2.55,-0.39,True
3,outperform_rate,Outperform rate,18,3.40,-7.72,15.43,False
4,downside_5pct_rate,5% downside-event rate,18,13.58,6.17,22.53,True
5,downside_10pct_rate,10% downside-event rate,18,8.33,1.54,16.67,True



Attractive-minus-Unfavourable reference comparison:


,metric,difference_delta,lower_delta,upper_delta,matches_reference
0,future_return,-0.0013,-0.0040,-0.0001,True
1,excess_return,-0.0045,0.0009,-0.0012,True
2,worst_path_return,-0.0044,0.0027,0.0030,True
3,outperform_rate,-0.0002,0.0024,-0.0049,True
4,downside_5pct_rate,-0.0025,0.0044,-0.0024,True
5,downside_10pct_rate,-0.0042,0.0035,-0.0028,True



Q1-minus-Q5 reference comparison:


,metric,difference_delta,lower_delta,upper_delta,matches_reference
0,future_return,-0.0012,0.0011,0.0039,True
1,excess_return,-0.0034,0.0006,0.0026,True
2,worst_path_return,0.0003,0.0029,0.0013,True
3,downside_5pct_rate,0.0002,0.0028,0.0009,True
4,downside_10pct_rate,0.0033,0.0032,-0.0033,True



Paired bootstrap and Notebook 08 reference validation passed.


## Bootstrap Verification Summary

The paired bootstrap analysis tests whether differences between historical groups were consistent across the **18 monthly rebalance periods**, rather than being caused by a few unusual months.

### Bootstrap Configuration

- **Iterations:** 10,000
- **Random seed:** 42
- **Paired monthly observations:** 18
- **Confidence level:** 95%

For each metric, the difference between the two groups was calculated separately for every month. The 18 monthly differences were then repeatedly resampled to estimate uncertainty around the average difference.

A confidence interval that **includes zero** means no reliable difference was established. A confidence interval entirely above or below zero indicates a statistically meaningful difference at the 95% confidence level.

---

## Attractive Risk-Reward Versus Unfavourable Risk-Reward

The comparison is:

> **Attractive Risk-Reward − Unfavourable Risk-Reward**

| Metric | Difference | 95% confidence interval | Reliable difference? |
|---|---:|---:|:---:|
| Future return | −0.02 pp | −2.25 to +1.99 pp | No |
| Excess return | −0.02 pp | −2.26 to +1.99 pp | No |
| Worst-path return | +1.84 pp | +0.73 to +3.14 pp | Yes |
| Outperform rate | +3.73 pp | −10.33 to +17.83 pp | No |
| 5% downside-event rate | −13.70 pp | −24.59 to −3.98 pp | Yes |
| 10% downside-event rate | −9.60 pp | −19.70 to −1.99 pp | Yes |

### Interpretation

Attractive and Unfavourable stocks produced almost identical final 20-day returns.

The confidence intervals for future return, excess return, and outperform rate all cross zero. Therefore, there is no reliable evidence that Attractive stocks generated higher returns or stronger alpha.

The downside-risk results were statistically meaningful:

- Attractive stocks had drawdowns that were approximately **1.84 percentage points shallower**.
- Attractive stocks experienced **13.70 percentage points fewer 5% downside events**.
- Attractive stocks experienced **9.60 percentage points fewer 10% downside events**.
- All three risk confidence intervals excluded zero.

The combined classification therefore provides reliable **downside-risk separation**, but not reliable return separation.

---

## Highest-Risk Versus Lowest-Risk Quintiles

The comparison is:

> **Q1 Highest Risk − Q5 Lowest Risk**

| Metric | Difference | 95% confidence interval | Reliable difference? |
|---|---:|---:|:---:|
| Future return | +1.33 pp | −0.73 to +3.46 pp | No |
| Excess return | +1.34 pp | −0.72 to +3.47 pp | No |
| Worst-path return | −1.45 pp | −2.55 to −0.39 pp | Yes |
| Outperform rate | +3.40 pp | −7.72 to +15.43 pp | No |
| 5% downside-event rate | +13.58 pp | +6.17 to +22.53 pp | Yes |
| 10% downside-event rate | +8.33 pp | +1.54 to +16.67 pp | Yes |

### Interpretation

The highest-risk quintile produced higher observed final and excess returns, but both confidence intervals crossed zero. The apparent return advantage was therefore not statistically reliable.

The downside outcomes showed reliable separation:

- Q1 stocks experienced drawdowns that were approximately **1.45 percentage points deeper** than Q5 stocks.
- Q1 stocks experienced **13.58 percentage points more 5% downside events**.
- Q1 stocks experienced **8.33 percentage points more 10% downside events**.
- All three downside-risk confidence intervals excluded zero.

This confirms that the downside model ranks future drawdown exposure, even though it does not reliably rank final returns.

---

## Reference Reproduction

Notebook 10 independently reproduced the Notebook 08 bootstrap results.

Every reference comparison returned:

> **matches_reference = True**

The newly calculated point estimates and confidence intervals differed from the Notebook 08 values by only tiny fractions of one percentage point. These differences are caused by full-precision calculations and displayed rounding.

This confirms that Notebook 10 reproduced the same:

- Historical monthly sample
- Fixed trained models
- Ordered model-feature list
- Within-date ranking procedure
- Opportunity-tier boundaries
- Downside-risk quintile boundaries
- 45/45 relative-risk split
- Combined classification rules
- Paired bootstrap procedure
- Bootstrap iteration count and random seed

---

## Statistical Findings

### Statistically Supported

The rank-based system reliably separates stocks based on:

- Average worst-path drawdown
- Frequency of 5% downside events
- Frequency of 10% downside events

Both the Attractive-versus-Unfavourable comparison and the Q1-versus-Q5 comparison showed meaningful downside-risk separation.

### Not Statistically Supported

The current backtest does not establish reliable group differences in:

- Final 20-day return
- Excess return versus Nifty 50
- Benchmark-outperformance rate

The observed return differences may have resulted from normal monthly sampling variation.

---

## Verification Conclusion

The exact rank-based classification implemented in Notebook 09 has been independently reproduced and historically verified.

The evidence supports MarketGuard as a:

> **Relative downside-risk ranking and stock-screening framework**

The evidence does not support presenting it as a:

> **Proven alpha model, market-beating strategy, or direct buy/sell system**

The primary value of the system is identifying stocks with relatively higher or lower future drawdown exposure while keeping opportunity ranking and downside risk as separate signals.

### Leakage audit and v1-versus-v2 stability comparison

In [7]:
# Audit leakage and compare fixed-band versus rank-based stability

verification_audit = historical_scored.copy()


# ---------------------------------------------------------
# 1. Look-ahead leakage audit
# ---------------------------------------------------------

outcome_feature_overlap = sorted(set(MODEL_FEATURES).intersection(OUTCOME_COLUMNS))

suspicious_feature_names = [
    feature
    for feature in MODEL_FEATURES
    if feature.lower().startswith(("future_", "target_"))
]

if outcome_feature_overlap:
    raise ValueError(
        "Future outcome columns were found in the model features:\n"
        + "\n".join(outcome_feature_overlap)
    )

if suspicious_feature_names:
    raise ValueError(
        "Potential future or target columns were found in the model features:\n"
        + "\n".join(suspicious_feature_names)
    )

if list(X_historical.columns) != MODEL_FEATURES:
    raise ValueError("The historical scoring matrix does not match the fitted feature order.")


# ---------------------------------------------------------
# 2. Recreate the original v1 broad risk and combined class
# ---------------------------------------------------------

verification_audit["v1_broad_risk"] = np.where(
    verification_audit["downside_risk_band"].isin(["Low Risk", "Watch Risk"]),
    "Lower Fixed-Band Risk",
    "Higher Fixed-Band Risk",
)

verification_audit["v1_opportunity_risk_class"] = np.select(
    [
        verification_audit["opportunity_tier"].eq("High Opportunity")
        & verification_audit["v1_broad_risk"].eq("Lower Fixed-Band Risk"),

        verification_audit["opportunity_tier"].eq("High Opportunity")
        & verification_audit["v1_broad_risk"].eq("Higher Fixed-Band Risk"),

        verification_audit["opportunity_tier"].eq("Moderate Opportunity")
        & verification_audit["v1_broad_risk"].eq("Lower Fixed-Band Risk"),

        verification_audit["opportunity_tier"].eq("Moderate Opportunity")
        & verification_audit["v1_broad_risk"].eq("Higher Fixed-Band Risk"),

        verification_audit["opportunity_tier"].eq("Low Opportunity")
        & verification_audit["v1_broad_risk"].eq("Lower Fixed-Band Risk"),
    ],
    [
        "Attractive Risk-Reward",
        "High Opportunity / High Risk",
        "Balanced Opportunity",
        "Caution",
        "Low Opportunity / Lower Risk",
    ],
    default="Unfavourable Risk-Reward",
)

verification_audit["v1_v2_class_changed"] = (
    verification_audit["v1_opportunity_risk_class"]
    != verification_audit["opportunity_risk_class"]
)


# ---------------------------------------------------------
# 3. Compare broad risk-group size stability
# ---------------------------------------------------------

v1_risk_counts_by_date = pd.crosstab(
    verification_audit["date"],
    verification_audit["v1_broad_risk"],
)

v2_risk_counts_by_date = pd.crosstab(
    verification_audit["date"],
    verification_audit["relative_downside_risk"],
)


def summarize_group_sizes(count_table: pd.DataFrame, method: str) -> pd.DataFrame:
    records = []

    for group in count_table.columns:
        values = count_table[group]

        records.append({
            "method": method,
            "risk_group": group,
            "mean_stocks": values.mean(),
            "std_stocks": values.std(ddof=0),
            "minimum_stocks": values.min(),
            "maximum_stocks": values.max(),
        })

    return pd.DataFrame(records)


risk_group_stability = pd.concat(
    [
        summarize_group_sizes(v1_risk_counts_by_date, "V1 fixed probability bands"),
        summarize_group_sizes(v2_risk_counts_by_date, "V2 relative ranks"),
    ],
    ignore_index=True,
)


# ---------------------------------------------------------
# 4. Compare historical v1 and v2 risk separation
# ---------------------------------------------------------

clean_audit = verification_audit.loc[
    verification_audit["date"] >= CLEAN_TEST_START_DATE
].copy()

monthly_v1_performance = calculate_monthly_group_performance(
    clean_audit,
    "v1_opportunity_risk_class",
)

_, v1_attractive_unfavourable_spread = calculate_paired_spread(
    monthly_v1_performance,
    "v1_opportunity_risk_class",
    "Attractive Risk-Reward",
    "Unfavourable Risk-Reward",
)

v1_spread_lookup = v1_attractive_unfavourable_spread.set_index("metric")[
    "difference_percentage_points"
]

v2_spread_lookup = attractive_unfavourable_spread.set_index("metric")[
    "difference_percentage_points"
]

comparison_metrics = [
    "future_return",
    "worst_path_return",
    "downside_5pct_rate",
    "downside_10pct_rate",
]

v1_v2_spread_comparison = pd.DataFrame({
    "metric": comparison_metrics,
    "v1_fixed_band_difference": [
        v1_spread_lookup[metric] for metric in comparison_metrics
    ],
    "v2_rank_based_difference": [
        v2_spread_lookup[metric] for metric in comparison_metrics
    ],
})

v1_v2_spread_comparison["v2_minus_v1"] = (
    v1_v2_spread_comparison["v2_rank_based_difference"]
    - v1_v2_spread_comparison["v1_fixed_band_difference"]
)


# ---------------------------------------------------------
# 5. Classification disagreement
# ---------------------------------------------------------

class_changes_by_date = (
    verification_audit.groupby("date")["v1_v2_class_changed"]
    .agg(["sum", "mean"])
    .rename(columns={"sum": "changed_stocks", "mean": "changed_rate"})
    .reset_index()
)

class_changes_by_date["changed_rate"] *= 100

overall_changed_stocks = int(verification_audit["v1_v2_class_changed"].sum())
overall_changed_rate = verification_audit["v1_v2_class_changed"].mean() * 100


# ---------------------------------------------------------
# 6. Display results
# ---------------------------------------------------------

print("Look-ahead leakage audit:")
print("Model features:", len(MODEL_FEATURES))
print("Outcome-feature overlap:", len(outcome_feature_overlap))
print("Suspicious future/target feature names:", len(suspicious_feature_names))
print("Historical scoring feature order matches fitted models:", True)

print("\nBroad risk-group size stability:")
display(risk_group_stability.round(2))

print("\nV1 fixed-band versus V2 rank-based Attractive-minus-Unfavourable spread:")
display(v1_v2_spread_comparison.round(2))

print("\nHistorical V1-to-V2 classification changes:")
print("Changed stock-month observations:", overall_changed_stocks)
print("Changed rate:", f"{overall_changed_rate:.2f}%")
print("Minimum changes in a month:", int(class_changes_by_date["changed_stocks"].min()))
print("Maximum changes in a month:", int(class_changes_by_date["changed_stocks"].max()))
print("Average changes per month:", f"{class_changes_by_date['changed_stocks'].mean():.2f}")

print("\nMonthly classification-change summary:")

class_changes_display = class_changes_by_date.copy()
class_changes_display["changed_rate"] = class_changes_display["changed_rate"].round(2)
display(class_changes_display)

print("\nLeakage and method-stability validation passed.")

Look-ahead leakage audit:
Model features: 79
Outcome-feature overlap: 0
Suspicious future/target feature names: 0
Historical scoring feature order matches fitted models: True

Broad risk-group size stability:


,method,risk_group,mean_stocks,std_stocks,minimum_stocks,maximum_stocks
0,V1 fixed probability bands,Higher Fixed-Band Risk,29.87,23.48,1,87
1,V1 fixed probability bands,Lower Fixed-Band Risk,60.13,23.48,3,89
2,V2 relative ranks,Higher Relative Risk,45.00,0.00,45,45
3,V2 relative ranks,Lower Relative Risk,45.00,0.00,45,45



V1 fixed-band versus V2 rank-based Attractive-minus-Unfavourable spread:


,metric,v1_fixed_band_difference,v2_rank_based_difference,v2_minus_v1
0,future_return,-0.05,-0.02,0.03
1,worst_path_return,1.86,1.84,-0.02
2,downside_5pct_rate,-17.20,-13.70,3.50
3,downside_10pct_rate,-7.53,-9.60,-2.08



Historical V1-to-V2 classification changes:
Changed stock-month observations: 738
Changed rate: 27.33%
Minimum changes in a month: 0
Maximum changes in a month: 44
Average changes per month: 24.60

Monthly classification-change summary:


,date,changed_stocks,changed_rate
0,2024-01-02,37,41.11
1,2024-02-01,13,14.44
2,2024-03-01,22,24.44
3,2024-04-01,7,7.78
4,2024-05-02,29,32.22
5,2024-06-03,24,26.67
6,2024-07-01,31,34.44
7,2024-08-01,21,23.33
8,2024-09-02,30,33.33
9,2024-10-01,18,20.00



Leakage and method-stability validation passed.


## Leakage and Method-Stability Observations

### Look-Ahead Leakage Audit

The historical scoring process passed the direct leakage checks:

- The two models use **79 input features**.
- None of the future outcome columns appear in the model-feature list.
- No model feature begins with `future_` or `target_`.
- The historical scoring matrix uses the same ordered feature list stored in the fitted models.

This confirms that future returns, downside outcomes, and target labels were not directly supplied to the models during historical scoring.

The audit verifies the inputs used in this notebook. It does not independently reconstruct and audit every upstream feature-generation step.

---

## Broad Risk-Group Stability

| Method | Risk group | Average stocks | Standard deviation | Minimum | Maximum |
|---|---|---:|---:|---:|---:|
| V1 fixed probability bands | Higher Fixed-Band Risk | 29.87 | 23.48 | 1 | 87 |
| V1 fixed probability bands | Lower Fixed-Band Risk | 60.13 | 23.48 | 3 | 89 |
| V2 relative ranks | Higher Relative Risk | 45.00 | 0.00 | 45 | 45 |
| V2 relative ranks | Lower Relative Risk | 45.00 | 0.00 | 45 | 45 |

### Observation

The V1 fixed-band groups changed dramatically across historical market periods.

Depending on the month, the Higher Fixed-Band Risk group contained anywhere from **1 to 87 stocks**. This occurred because the distribution of raw downside probabilities shifted over time while the probability thresholds remained fixed.

The V2 rank-based design always assigned:

- 45 stocks to Higher Relative Risk
- 45 stocks to Lower Relative Risk

This gives V2 a stable cross-sectional interpretation across different market regimes.

The 45/45 split does not mean that exactly half the market is absolutely dangerous. It means that half of the eligible stocks are relatively riskier than the other half on the same rebalance date.

---

## V1 Versus V2 Historical Risk Separation

The table compares Attractive Risk-Reward minus Unfavourable Risk-Reward results under the two classification methods.

| Metric | V1 fixed bands | V2 relative ranks | V2 minus V1 |
|---|---:|---:|---:|
| Future return | −0.05 pp | −0.02 pp | +0.03 pp |
| Worst-path return | +1.86 pp | +1.84 pp | −0.02 pp |
| 5% downside-event rate | −17.20 pp | −13.70 pp | +3.50 pp |
| 10% downside-event rate | −7.53 pp | −9.60 pp | −2.08 pp |

### Observation

Both methods produced almost identical conclusions for final return and worst-path drawdown.

- Neither method showed a meaningful Attractive-versus-Unfavourable final-return advantage.
- V1 and V2 both showed approximately **1.8 percentage points shallower drawdowns** for Attractive stocks.
- V1 showed stronger separation for 5% downside events.
- V2 showed stronger separation for 10% downside events.

These point estimates do not establish that V2 is statistically superior to V1.

The main reason to prefer V2 is its stable and consistent group construction while retaining meaningful downside-risk separation.

---

## Historical Classification Changes

Across the complete historical verification sample:

- Total stock-month observations: **2,700**
- Changed V1-to-V2 classifications: **738**
- Overall changed rate: **27.33%**
- Average changes per month: **24.60 stocks**
- Minimum monthly changes: **0**
- Maximum monthly changes: **44**

### Observation

V1 and V2 assigned different combined classifications to more than one-quarter of all historical stock-month observations.

The amount of disagreement varied significantly by month:

- Some months had almost no changes.
- Several months had more than 40 changes.
- The largest monthly disagreement was 44 of 90 stocks, or 48.89%.

The disagreement became particularly large during parts of late 2025 and early 2026. During these periods, the fixed probability thresholds and relative ranks represented substantially different market-risk splits.

This shows that the small five-stock difference observed in the latest July 2026 snapshot is not representative of all historical periods.

---

## Methodological Conclusion

The V2 rank-based method should remain the primary production classification because it:

- Uses the same leakage-free model inputs.
- Maintains stable 45/45 relative-risk groups.
- Avoids extreme group-size changes caused by raw probability drift.
- Preserves statistically meaningful downside-risk separation.
- Provides a consistent interpretation across changing market regimes.

V2 is not preferred because it produces higher final returns.

It is preferred because it provides a more stable, interpretable, and historically validated relative downside-risk screening framework.

The fixed raw-probability bands can still be retained as descriptive alert levels, but they should not determine the primary combined opportunity-risk classification.

### Save verification artifacts and metadata

In [8]:
# Save verification artifacts, metadata, and report summary

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

AUDIT_COLUMNS = [
    "date",
    "yf_ticker",
    "outperform_probability",
    "downside_probability",
    "outperform_rank",
    "downside_risk_rank",
    "opportunity_tier",
    "downside_risk_band",
    "downside_risk_quintile",
    "relative_downside_risk",
    "opportunity_risk_class",
    "v1_broad_risk",
    "v1_opportunity_risk_class",
    "v1_v2_class_changed",
    *OUTCOME_COLUMNS,
]

verification_results = (
    verification_audit[AUDIT_COLUMNS]
    .sort_values(["date", "yf_ticker"])
    .reset_index(drop=True)
)


# ---------------------------------------------------------
# Output paths
# ---------------------------------------------------------

verification_csv_path = OUTPUT_DIR / "historical_rank_based_verification_v1.csv"
verification_parquet_path = OUTPUT_DIR / "historical_rank_based_verification_v1.parquet"

rebalance_dates_path = OUTPUT_DIR / "selected_rebalance_dates_v1.csv"
class_summary_path = OUTPUT_DIR / "combined_class_summary_2025plus_v1.csv"
quintile_summary_path = OUTPUT_DIR / "risk_quintile_summary_2025plus_v1.csv"

combined_bootstrap_path = OUTPUT_DIR / "attractive_unfavourable_bootstrap_v1.csv"
quintile_bootstrap_path = OUTPUT_DIR / "q1_q5_bootstrap_v1.csv"

method_stability_path = OUTPUT_DIR / "v1_v2_method_stability_v1.csv"
spread_comparison_path = OUTPUT_DIR / "v1_v2_spread_comparison_v1.csv"
monthly_changes_path = OUTPUT_DIR / "monthly_classification_changes_v1.csv"

metadata_path = OUTPUT_DIR / "rank_based_verification_metadata_v1.json"
summary_path = OUTPUT_DIR / "rank_based_verification_summary_v1.md"


# ---------------------------------------------------------
# Save tabular artifacts
# ---------------------------------------------------------

verification_results.to_csv(verification_csv_path, index=False)
verification_results.to_parquet(verification_parquet_path, index=False)

selected_rebalance_dates.to_csv(rebalance_dates_path, index=False)
class_summary_2025plus.to_csv(class_summary_path, index=False)
quintile_summary_2025plus.to_csv(quintile_summary_path, index=False)

attractive_unfavourable_bootstrap.to_csv(combined_bootstrap_path, index=False)
highest_lowest_risk_bootstrap.to_csv(quintile_bootstrap_path, index=False)

risk_group_stability.to_csv(method_stability_path, index=False)
v1_v2_spread_comparison.to_csv(spread_comparison_path, index=False)
class_changes_by_date.to_csv(monthly_changes_path, index=False)


# ---------------------------------------------------------
# Extract important verified results
# ---------------------------------------------------------

combined_bootstrap_lookup = attractive_unfavourable_bootstrap.set_index("metric")
quintile_bootstrap_lookup = highest_lowest_risk_bootstrap.set_index("metric")


def bootstrap_result(table: pd.DataFrame, metric: str) -> dict:
    row = table.loc[metric]

    return {
        "difference_percentage_points": float(row["difference_percentage_points"]),
        "ci_95_lower": float(row["ci_95_lower"]),
        "ci_95_upper": float(row["ci_95_upper"]),
        "ci_excludes_zero": bool(row["ci_excludes_zero"]),
    }


def dataframe_records(dataframe: pd.DataFrame) -> list[dict]:
    return json.loads(dataframe.to_json(orient="records", date_format="iso"))


# ---------------------------------------------------------
# Save verification metadata
# ---------------------------------------------------------

verification_metadata = {
    "notebook": "10_rank_based_snapshot_verification_backtest",
    "notebook_version": NOTEBOOK_VERSION,
    "verification_version": VERIFICATION_VERSION,
    "classification_method": CLASSIFICATION_METHOD,
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "source_files": {
        "feature_dataset": str(FEATURE_DATA_PATH.relative_to(PROJECT_ROOT)),
        "outperform_model": str(OUTPERFORM_MODEL_PATH.relative_to(PROJECT_ROOT)),
        "downside_model": str(DOWNSIDE_MODEL_PATH.relative_to(PROJECT_ROOT)),
        "notebook_08_metadata": str(NOTEBOOK_08_METADATA_PATH.relative_to(PROJECT_ROOT)),
        "notebook_09_metadata": str(NOTEBOOK_09_METADATA_PATH.relative_to(PROJECT_ROOT)),
    },
    "model_configuration": {
        "feature_count": len(MODEL_FEATURES),
        "outperform_classes": outperform_classes.tolist(),
        "downside_classes": downside_classes.tolist(),
        "identical_ordered_feature_list": same_feature_order,
    },
    "historical_sample": {
        "start_date": verification_results["date"].min().date().isoformat(),
        "end_date": verification_results["date"].max().date().isoformat(),
        "rebalance_dates": int(verification_results["date"].nunique()),
        "rows": len(verification_results),
        "stocks_per_rebalance": EXPECTED_STOCKS_PER_REBALANCE,
    },
    "clean_test_period": {
        "start_date": clean_scored["date"].min().date().isoformat(),
        "end_date": clean_scored["date"].max().date().isoformat(),
        "rebalance_dates": int(clean_scored["date"].nunique()),
        "rows": len(clean_scored),
    },
    "bootstrap_configuration": {
        "iterations": BOOTSTRAP_ITERATIONS,
        "random_seed": BOOTSTRAP_RANDOM_SEED,
        "paired_months": len(attractive_unfavourable_monthly_spread),
        "confidence_level": 0.95,
    },
    "leakage_audit": {
        "outcome_feature_overlap_count": len(outcome_feature_overlap),
        "suspicious_feature_name_count": len(suspicious_feature_names),
        "feature_order_matches_fitted_models": True,
        "direct_leakage_check_passed": True,
    },
    "reference_reproduction": {
        "combined_class_results_match_notebook_08": bool(
            combined_reference_check["matches_reference"].all()
        ),
        "risk_quintile_results_match_notebook_08": bool(
            quintile_reference_check["matches_reference"].all()
        ),
    },
    "attractive_minus_unfavourable": {
        "future_return": bootstrap_result(combined_bootstrap_lookup, "future_return"),
        "excess_return": bootstrap_result(combined_bootstrap_lookup, "excess_return"),
        "worst_path_return": bootstrap_result(
            combined_bootstrap_lookup,
            "worst_path_return",
        ),
        "downside_5pct_rate": bootstrap_result(
            combined_bootstrap_lookup,
            "downside_5pct_rate",
        ),
        "downside_10pct_rate": bootstrap_result(
            combined_bootstrap_lookup,
            "downside_10pct_rate",
        ),
    },
    "q1_highest_minus_q5_lowest": {
        "future_return": bootstrap_result(quintile_bootstrap_lookup, "future_return"),
        "excess_return": bootstrap_result(quintile_bootstrap_lookup, "excess_return"),
        "worst_path_return": bootstrap_result(
            quintile_bootstrap_lookup,
            "worst_path_return",
        ),
        "downside_5pct_rate": bootstrap_result(
            quintile_bootstrap_lookup,
            "downside_5pct_rate",
        ),
        "downside_10pct_rate": bootstrap_result(
            quintile_bootstrap_lookup,
            "downside_10pct_rate",
        ),
    },
    "method_stability": {
        "risk_group_sizes": dataframe_records(risk_group_stability),
        "changed_stock_month_observations": overall_changed_stocks,
        "changed_rate_percent": float(overall_changed_rate),
        "average_changes_per_month": float(
            class_changes_by_date["changed_stocks"].mean()
        ),
        "minimum_changes_per_month": int(
            class_changes_by_date["changed_stocks"].min()
        ),
        "maximum_changes_per_month": int(
            class_changes_by_date["changed_stocks"].max()
        ),
    },
    "final_interpretation": {
        "validated_use": "relative downside-risk ranking and stock screening",
        "not_validated_use": "reliable alpha, market-beating, or direct buy/sell prediction",
        "recommended_primary_classification": "v2 rank-based relative downside risk",
        "fixed_probability_bands_role": "descriptive alert levels only",
    },
}

with metadata_path.open("w", encoding="utf-8") as file:
    json.dump(verification_metadata, file, indent=2, ensure_ascii=False)


# ---------------------------------------------------------
# Save concise Markdown report
# ---------------------------------------------------------

summary_markdown = f"""# Rank-Based Snapshot Verification Backtest

## Verification Scope

- Historical period: {verification_results["date"].min().date()} to {verification_results["date"].max().date()}
- Monthly rebalance dates: {verification_results["date"].nunique()}
- Stocks per rebalance date: {EXPECTED_STOCKS_PER_REBALANCE}
- Historical observations: {len(verification_results)}
- Clean evaluation period: {clean_scored["date"].min().date()} to {clean_scored["date"].max().date()}
- Clean monthly observations: {clean_scored["date"].nunique()}

## Reproduction Result

Notebook 10 independently reproduced the Notebook 08 point estimates and
paired-bootstrap confidence intervals.

- Combined-class reference match: **{bool(combined_reference_check["matches_reference"].all())}**
- Risk-quintile reference match: **{bool(quintile_reference_check["matches_reference"].all())}**

## Leakage Audit

- Model features: **{len(MODEL_FEATURES)}**
- Outcome-feature overlap: **{len(outcome_feature_overlap)}**
- Suspicious future or target feature names: **{len(suspicious_feature_names)}**
- Historical feature order matches fitted models: **True**

The verification scoring process passed the direct target-leakage audit.

## Attractive Versus Unfavourable

- Future-return difference: **{combined_bootstrap_lookup.loc["future_return", "difference_percentage_points"]:.2f} pp**
- Worst-path difference: **{combined_bootstrap_lookup.loc["worst_path_return", "difference_percentage_points"]:.2f} pp**
- 5% downside-event difference: **{combined_bootstrap_lookup.loc["downside_5pct_rate", "difference_percentage_points"]:.2f} pp**
- 10% downside-event difference: **{combined_bootstrap_lookup.loc["downside_10pct_rate", "difference_percentage_points"]:.2f} pp**

The worst-path and downside-event confidence intervals exclude zero. The
future-return and excess-return confidence intervals include zero.

## Highest-Risk Versus Lowest-Risk Quintiles

- Future-return difference: **{quintile_bootstrap_lookup.loc["future_return", "difference_percentage_points"]:.2f} pp**
- Worst-path difference: **{quintile_bootstrap_lookup.loc["worst_path_return", "difference_percentage_points"]:.2f} pp**
- 5% downside-event difference: **{quintile_bootstrap_lookup.loc["downside_5pct_rate", "difference_percentage_points"]:.2f} pp**
- 10% downside-event difference: **{quintile_bootstrap_lookup.loc["downside_10pct_rate", "difference_percentage_points"]:.2f} pp**

The downside model reliably separates future drawdown exposure but does not
reliably separate final returns.

## Method Stability

The V1 fixed probability bands produced highly variable broad risk-group sizes:

- Higher Fixed-Band Risk: **1 to 87 stocks**
- Lower Fixed-Band Risk: **3 to 89 stocks**

The V2 rank-based method maintained:

- Higher Relative Risk: **45 stocks**
- Lower Relative Risk: **45 stocks**

V1 and V2 classifications differed for **{overall_changed_stocks} of
{len(verification_results)} stock-month observations
({overall_changed_rate:.2f}%)**.

## Final Conclusion

The exact Notebook 09 implementation is historically verified as a
**relative downside-risk ranking and stock-screening framework**.

It is not validated as a reliable alpha model, market-beating strategy, or
direct buy/sell system.

The V2 rank-based method should remain the primary combined classification.
Fixed raw-probability bands should remain descriptive alert levels only.
"""

summary_path.write_text(summary_markdown, encoding="utf-8")


# ---------------------------------------------------------
# Confirm saved artifacts
# ---------------------------------------------------------

saved_paths = [
    verification_csv_path,
    verification_parquet_path,
    rebalance_dates_path,
    class_summary_path,
    quintile_summary_path,
    combined_bootstrap_path,
    quintile_bootstrap_path,
    method_stability_path,
    spread_comparison_path,
    monthly_changes_path,
    metadata_path,
    summary_path,
]

missing_saved_paths = [path for path in saved_paths if not path.exists()]

if missing_saved_paths:
    raise FileNotFoundError(
        "The following artifacts were not saved:\n"
        + "\n".join(str(path) for path in missing_saved_paths)
    )

print("Saved verification artifacts:", len(saved_paths))
print("Output directory:", OUTPUT_DIR)

for path in saved_paths:
    print(f"[SAVED] {path.name}")

print("\nVerification artifacts and report metadata saved successfully.")

Saved verification artifacts: 12
Output directory: E:\Projects\marketguard-india\reports\rank_based_snapshot_verification_backtest
[SAVED] historical_rank_based_verification_v1.csv
[SAVED] historical_rank_based_verification_v1.parquet
[SAVED] selected_rebalance_dates_v1.csv
[SAVED] combined_class_summary_2025plus_v1.csv
[SAVED] risk_quintile_summary_2025plus_v1.csv
[SAVED] attractive_unfavourable_bootstrap_v1.csv
[SAVED] q1_q5_bootstrap_v1.csv
[SAVED] v1_v2_method_stability_v1.csv
[SAVED] v1_v2_spread_comparison_v1.csv
[SAVED] monthly_classification_changes_v1.csv
[SAVED] rank_based_verification_metadata_v1.json
[SAVED] rank_based_verification_summary_v1.md

Verification artifacts and report metadata saved successfully.


### Independent artifact reload and validation

In [9]:
# Independently reload and validate all saved artifacts

reloaded_verification_csv = pd.read_csv(verification_csv_path)
reloaded_verification_parquet = pd.read_parquet(verification_parquet_path)

reloaded_rebalance_dates = pd.read_csv(rebalance_dates_path)
reloaded_class_summary = pd.read_csv(class_summary_path)
reloaded_quintile_summary = pd.read_csv(quintile_summary_path)

reloaded_combined_bootstrap = pd.read_csv(combined_bootstrap_path)
reloaded_quintile_bootstrap = pd.read_csv(quintile_bootstrap_path)

reloaded_method_stability = pd.read_csv(method_stability_path)
reloaded_spread_comparison = pd.read_csv(spread_comparison_path)
reloaded_monthly_changes = pd.read_csv(monthly_changes_path)

with metadata_path.open("r", encoding="utf-8") as file:
    reloaded_metadata = json.load(file)

reloaded_summary = summary_path.read_text(encoding="utf-8")


# ---------------------------------------------------------
# Normalize CSV and Parquet values before comparison
# ---------------------------------------------------------

reloaded_verification_csv["date"] = pd.to_datetime(
    reloaded_verification_csv["date"]
)

reloaded_verification_parquet["date"] = pd.to_datetime(
    reloaded_verification_parquet["date"]
)

reloaded_rebalance_dates["rebalance_date"] = pd.to_datetime(
    reloaded_rebalance_dates["rebalance_date"]
)

reloaded_monthly_changes["date"] = pd.to_datetime(
    reloaded_monthly_changes["date"]
)

reloaded_verification_csv = (
    reloaded_verification_csv
    .sort_values(["date", "yf_ticker"])
    .reset_index(drop=True)
)

reloaded_verification_parquet = (
    reloaded_verification_parquet
    .sort_values(["date", "yf_ticker"])
    .reset_index(drop=True)
)


pd.testing.assert_frame_equal(
    reloaded_verification_csv,
    reloaded_verification_parquet,
    check_dtype=False,
    check_exact=False,
    rtol=1e-10,
    atol=1e-10,
)


# ---------------------------------------------------------
# Main historical verification artifact
# ---------------------------------------------------------

expected_verification_shape = (2_700, len(AUDIT_COLUMNS))

if reloaded_verification_csv.shape != expected_verification_shape:
    raise ValueError(
        f"Expected verification shape {expected_verification_shape}, "
        f"found {reloaded_verification_csv.shape}."
    )

if reloaded_verification_csv["date"].nunique() != 30:
    raise ValueError("Expected 30 rebalance dates in the reloaded verification data.")

if reloaded_verification_csv["yf_ticker"].nunique() != 90:
    raise ValueError("Expected 90 unique stocks in the verification sample.")

if reloaded_verification_csv.duplicated(["date", "yf_ticker"]).any():
    raise ValueError("Reloaded verification data contains duplicate stock/date rows.")

reloaded_stocks_per_date = (
    reloaded_verification_csv.groupby("date")["yf_ticker"].nunique()
)

if not reloaded_stocks_per_date.eq(90).all():
    raise ValueError("Not every reloaded rebalance date contains exactly 90 stocks.")

if reloaded_verification_csv["date"].min() != pd.Timestamp("2024-01-02"):
    raise ValueError("Unexpected historical verification start date.")

if reloaded_verification_csv["date"].max() != pd.Timestamp("2026-06-01"):
    raise ValueError("Unexpected historical verification end date.")


# ---------------------------------------------------------
# Rank-based structure
# ---------------------------------------------------------

reloaded_relative_counts = pd.crosstab(
    reloaded_verification_csv["date"],
    reloaded_verification_csv["relative_downside_risk"],
)

for label in ["Higher Relative Risk", "Lower Relative Risk"]:
    if not reloaded_relative_counts[label].eq(45).all():
        raise ValueError(f"{label} does not contain 45 stocks on every date.")

reloaded_quintile_counts = pd.crosstab(
    reloaded_verification_csv["date"],
    reloaded_verification_csv["downside_risk_quintile"],
)

expected_quintiles = [
    "Q1 — Highest Risk",
    "Q2 — High Risk",
    "Q3 — Medium Risk",
    "Q4 — Lower Risk",
    "Q5 — Lowest Risk",
]

for label in expected_quintiles:
    if not reloaded_quintile_counts[label].eq(18).all():
        raise ValueError(f"{label} does not contain 18 stocks on every date.")


# ---------------------------------------------------------
# Rebalance-date artifact
# ---------------------------------------------------------

if reloaded_rebalance_dates.shape != (30, 3):
    raise ValueError(
        f"Expected rebalance-date shape (30, 3), found {reloaded_rebalance_dates.shape}."
    )

april_2026_date = reloaded_rebalance_dates.loc[
    reloaded_rebalance_dates["month"].eq("2026-04"),
    "rebalance_date",
]

if len(april_2026_date) != 1:
    raise ValueError("Expected one selected rebalance date for April 2026.")

if april_2026_date.iloc[0] != pd.Timestamp("2026-04-02"):
    raise ValueError("April 2026 did not reload with the expected April 2 date.")


# ---------------------------------------------------------
# Summary-table shapes
# ---------------------------------------------------------

expected_shapes = {
    "combined class summary": (6, 9),
    "risk quintile summary": (5, 9),
    "combined bootstrap": (6, 7),
    "risk quintile bootstrap": (6, 7),
    "method stability": (4, 6),
    "spread comparison": (4, 4),
    "monthly classification changes": (30, 3),
}

actual_shapes = {
    "combined class summary": reloaded_class_summary.shape,
    "risk quintile summary": reloaded_quintile_summary.shape,
    "combined bootstrap": reloaded_combined_bootstrap.shape,
    "risk quintile bootstrap": reloaded_quintile_bootstrap.shape,
    "method stability": reloaded_method_stability.shape,
    "spread comparison": reloaded_spread_comparison.shape,
    "monthly classification changes": reloaded_monthly_changes.shape,
}

for artifact_name, expected_shape in expected_shapes.items():
    if actual_shapes[artifact_name] != expected_shape:
        raise ValueError(
            f"{artifact_name} expected shape {expected_shape}, "
            f"found {actual_shapes[artifact_name]}."
        )


# ---------------------------------------------------------
# Bootstrap conclusions
# ---------------------------------------------------------

expected_significant_metrics = {
    "worst_path_return",
    "downside_5pct_rate",
    "downside_10pct_rate",
}

combined_significant_metrics = set(
    reloaded_combined_bootstrap.loc[
        reloaded_combined_bootstrap["ci_excludes_zero"].astype(bool),
        "metric",
    ]
)

quintile_significant_metrics = set(
    reloaded_quintile_bootstrap.loc[
        reloaded_quintile_bootstrap["ci_excludes_zero"].astype(bool),
        "metric",
    ]
)

if combined_significant_metrics != expected_significant_metrics:
    raise ValueError(
        "Reloaded combined-class bootstrap significance results changed."
    )

if quintile_significant_metrics != expected_significant_metrics:
    raise ValueError(
        "Reloaded risk-quintile bootstrap significance results changed."
    )


# ---------------------------------------------------------
# Metadata validation
# ---------------------------------------------------------

if reloaded_metadata["verification_version"] != VERIFICATION_VERSION:
    raise ValueError("Unexpected verification version in metadata.")

if reloaded_metadata["classification_method"] != CLASSIFICATION_METHOD:
    raise ValueError("Unexpected classification method in metadata.")

if reloaded_metadata["model_configuration"]["feature_count"] != 79:
    raise ValueError("Metadata does not contain the expected 79 model features.")

if reloaded_metadata["historical_sample"]["rows"] != 2_700:
    raise ValueError("Metadata historical row count is incorrect.")

if reloaded_metadata["historical_sample"]["rebalance_dates"] != 30:
    raise ValueError("Metadata historical rebalance-date count is incorrect.")

if reloaded_metadata["clean_test_period"]["rebalance_dates"] != 18:
    raise ValueError("Metadata clean-period date count is incorrect.")

if reloaded_metadata["leakage_audit"]["outcome_feature_overlap_count"] != 0:
    raise ValueError("Metadata reports outcome-feature overlap.")

if reloaded_metadata["leakage_audit"]["suspicious_feature_name_count"] != 0:
    raise ValueError("Metadata reports suspicious future or target features.")

if not reloaded_metadata["leakage_audit"]["direct_leakage_check_passed"]:
    raise ValueError("Metadata direct leakage check did not pass.")

if not reloaded_metadata["reference_reproduction"][
    "combined_class_results_match_notebook_08"
]:
    raise ValueError("Combined-class results do not match Notebook 08.")

if not reloaded_metadata["reference_reproduction"][
    "risk_quintile_results_match_notebook_08"
]:
    raise ValueError("Risk-quintile results do not match Notebook 08.")

if reloaded_metadata["method_stability"][
    "changed_stock_month_observations"
] != 738:
    raise ValueError("Metadata V1-to-V2 changed-observation count is incorrect.")


# ---------------------------------------------------------
# Cross-check metadata against saved bootstrap tables
# ---------------------------------------------------------

def validate_metadata_bootstrap(
    metadata_section: dict,
    bootstrap_table: pd.DataFrame,
) -> None:
    bootstrap_lookup = bootstrap_table.set_index("metric")

    for metric, metadata_result in metadata_section.items():
        saved_result = bootstrap_lookup.loc[metric]

        numeric_fields = [
            "difference_percentage_points",
            "ci_95_lower",
            "ci_95_upper",
        ]

        for field in numeric_fields:
            if not np.isclose(
                metadata_result[field],
                saved_result[field],
                rtol=1e-10,
                atol=1e-10,
            ):
                raise ValueError(
                    f"Metadata and bootstrap CSV disagree for {metric}: {field}."
                )

        if bool(metadata_result["ci_excludes_zero"]) != bool(
            saved_result["ci_excludes_zero"]
        ):
            raise ValueError(
                f"Metadata and bootstrap CSV disagree for {metric}: "
                "ci_excludes_zero."
            )


validate_metadata_bootstrap(
    reloaded_metadata["attractive_minus_unfavourable"],
    reloaded_combined_bootstrap,
)

validate_metadata_bootstrap(
    reloaded_metadata["q1_highest_minus_q5_lowest"],
    reloaded_quintile_bootstrap,
)


# ---------------------------------------------------------
# Markdown summary validation
# ---------------------------------------------------------

required_summary_phrases = [
    "relative downside-risk ranking",
    "not validated as a reliable alpha model",
    "45 stocks",
    "Outcome-feature overlap",
]

missing_summary_phrases = [
    phrase for phrase in required_summary_phrases
    if phrase not in reloaded_summary
]

if missing_summary_phrases:
    raise ValueError(
        "The saved Markdown summary is missing required conclusions:\n"
        + "\n".join(missing_summary_phrases)
    )


# ---------------------------------------------------------
# Final output
# ---------------------------------------------------------

print("Reloaded artifact validation:")
print("Historical CSV:", reloaded_verification_csv.shape)
print("Historical Parquet:", reloaded_verification_parquet.shape)
print("CSV and Parquet equivalent:", True)
print("Rebalance-date artifact:", reloaded_rebalance_dates.shape)
print("Combined-class summary:", reloaded_class_summary.shape)
print("Risk-quintile summary:", reloaded_quintile_summary.shape)
print("Combined bootstrap:", reloaded_combined_bootstrap.shape)
print("Risk-quintile bootstrap:", reloaded_quintile_bootstrap.shape)
print("Method-stability table:", reloaded_method_stability.shape)
print("Spread-comparison table:", reloaded_spread_comparison.shape)
print("Monthly-change table:", reloaded_monthly_changes.shape)

print("\nHistorical structure:")
print("Rows:", len(reloaded_verification_csv))
print("Rebalance dates:", reloaded_verification_csv["date"].nunique())
print("Stocks per date:", int(reloaded_stocks_per_date.min()), "to", int(reloaded_stocks_per_date.max()))
print("Date range:", reloaded_verification_csv["date"].min().date(), "to", reloaded_verification_csv["date"].max().date())
print("April 2026 selected date:", april_2026_date.iloc[0].date())

print("\nBootstrap conclusions:")
print("Combined significant metrics:", sorted(combined_significant_metrics))
print("Quintile significant metrics:", sorted(quintile_significant_metrics))

print("\nMetadata:")
print("Verification version:", reloaded_metadata["verification_version"])
print("Classification method:", reloaded_metadata["classification_method"])
print("Direct leakage check passed:", reloaded_metadata["leakage_audit"]["direct_leakage_check_passed"])
print("Reference reproduction passed:", True)
print("Markdown summary characters:", len(reloaded_summary))

print("\nIndependent artifact reload validation passed.")

Reloaded artifact validation:
Historical CSV: (2700, 22)
Historical Parquet: (2700, 22)
CSV and Parquet equivalent: True
Rebalance-date artifact: (30, 3)
Combined-class summary: (6, 9)
Risk-quintile summary: (5, 9)
Combined bootstrap: (6, 7)
Risk-quintile bootstrap: (6, 7)
Method-stability table: (4, 6)
Spread-comparison table: (4, 4)
Monthly-change table: (30, 3)

Historical structure:
Rows: 2700
Rebalance dates: 30
Stocks per date: 90 to 90
Date range: 2024-01-02 to 2026-06-01
April 2026 selected date: 2026-04-02

Bootstrap conclusions:
Combined significant metrics: ['downside_10pct_rate', 'downside_5pct_rate', 'worst_path_return']
Quintile significant metrics: ['downside_10pct_rate', 'downside_5pct_rate', 'worst_path_return']

Metadata:
Verification version: v1
Classification method: rank_based_relative_downside_risk
Direct leakage check passed: True
Reference reproduction passed: True
Markdown summary characters: 2251

Independent artifact reload validation passed.
